# Linear Algebra Preliminaries for Numerical Linear Algebra

This notebook develops the foundational concepts needed before studying NLA proper. Each section pairs mathematical statements with computation and visualization. The goal is not to prove everything rigorously but to build the geometric and computational intuition that makes algorithms like QR, SVD, and Krylov methods intelligible.

**Contents**
1. The Four Fundamental Subspaces
2. Linear Independence, Span, Basis, Dimension
3. Linear Maps and Matrix Representations
4. Inner Products and Norms
5. Orthogonal Projections
6. Eigenvalues and the Spectral Theorem
7. Rank and Near-Rank-Deficiency

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from scipy import linalg

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})

rng = np.random.default_rng(42)
print('Environment ready.')

---
## 1. The Four Fundamental Subspaces

Every matrix $A \in \mathbb{R}^{m \times n}$ induces four subspaces:

| Subspace | Lives in | Definition |
|---|---|---|
| Column space $\mathcal{C}(A)$ | $\mathbb{R}^m$ | $\{Ax : x \in \mathbb{R}^n\}$ — range of $A$ |
| Null space $\mathcal{N}(A)$ | $\mathbb{R}^n$ | $\{x : Ax = 0\}$ |
| Row space $\mathcal{C}(A^T)$ | $\mathbb{R}^n$ | Column space of $A^T$ |
| Left null space $\mathcal{N}(A^T)$ | $\mathbb{R}^m$ | $\{y : A^T y = 0\}$ |

**Fundamental Theorem of Linear Algebra (Part 1):**
$$\dim \mathcal{C}(A) = \dim \mathcal{C}(A^T) = r \quad\text{(rank of }A)$$
$$\dim \mathcal{N}(A) = n - r, \qquad \dim \mathcal{N}(A^T) = m - r$$

**Fundamental Theorem (Part 2) — the orthogonality relations:**
$$\mathcal{C}(A^T) \perp \mathcal{N}(A) \quad \text{in } \mathbb{R}^n$$
$$\mathcal{C}(A) \perp \mathcal{N}(A^T) \quad \text{in } \mathbb{R}^m$$

*Proof sketch:* If $Ax = 0$ and $v = A^T w$ is any row space vector, then $v \cdot x = w^T A x = w^T 0 = 0$. $\square$

In [ ]:
# Concrete example: A is 3x4 of rank 2
# Construct A explicitly so we control the rank
U = np.array([[1, 0, 1],
              [0, 1, 0],
              [1, 1, 2]], dtype=float)  # 3x3, columns are C(A) basis + extra

A = np.array([[1, 0,  1, 2],
              [0, 1,  1, 1],
              [1, 1,  2, 3]], dtype=float)

print(f'A ({A.shape[0]}x{A.shape[1]}):')
print(A)
print(f'\nnp.linalg.matrix_rank(A) = {np.linalg.matrix_rank(A)}')

# --- Column space basis via QR with column pivoting ---
Q, R, P = linalg.qr(A, pivoting=True)  # AP = QR
r = np.sum(np.abs(np.diag(R)) > 1e-10)
col_space_basis = Q[:, :r]
print(f'\nColumn space basis (cols of Q from QR):')
print(col_space_basis)

# --- Null space via SVD ---
U_svd, S, Vt = np.linalg.svd(A)
null_space = Vt[r:, :].T  # rows of Vt corresponding to zero singular values
print(f'\nNull space basis (right singular vectors for sigma=0):')
print(null_space)

# Verify: A @ null_space should be ~0
print(f'\n||A @ null_space||_F = {np.linalg.norm(A @ null_space):.2e}  (should be ~0)')

# Verify orthogonality: C(A^T) ⊥ N(A)
row_space = Vt[:r, :].T  # right singular vectors for nonzero singular values
print(f'row_space^T @ null_space (should be ~0):')
print(np.round(row_space.T @ null_space, 10))

In [ ]:
# Visualize: for a 2x2 rank-1 matrix, show the four subspaces as lines/points
A2 = np.array([[2, 1],
               [4, 2]], dtype=float)  # rank 1: row2 = 2*row1

# Column space: span of [2,4]^T = span of [1,2]^T
# Null space: Ax=0 => 2x+y=0 => span of [1,-2]^T
# Row space: span of [2,1] = span of [2,1]
# Left null space: A^T y = 0 => 2y1+4y2=0, y1+2y2=0 => span of [2,-1]^T

_, _, Vt2 = np.linalg.svd(A2)
U2, _, _ = np.linalg.svd(A2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, title, space1, space2, label1, label2, color1, color2 in [
    (axes[0], r'$\mathbb{R}^2$ (domain): Row space $\perp$ Null space',
     Vt2[0], Vt2[1], r'$\mathcal{C}(A^T)$ row space', r'$\mathcal{N}(A)$ null space',
     '#2E86AB', '#E84855'),
    (axes[1], r'$\mathbb{R}^2$ (codomain): Column space $\perp$ Left null space',
     U2[:, 0], U2[:, 1], r'$\mathcal{C}(A)$ col space', r'$\mathcal{N}(A^T)$ left null',
     '#44BBA4', '#F7A278')
]:
    t = np.linspace(-2, 2, 100)
    ax.plot(t * space1[0], t * space1[1], color=color1, lw=2.5, label=label1)
    ax.plot(t * space2[0], t * space2[1], color=color2, lw=2.5, ls='--', label=label2)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    ax.set_title(title)
    # Annotate the right angle
    ax.annotate('⊥', xy=(0.1, 0.05), fontsize=16, color='gray')

plt.suptitle(r'Four Fundamental Subspaces of a $2\times 2$ rank-1 matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('four_subspaces.png', bbox_inches='tight')
plt.show()
print(f'Dot product of subspace directions: {space1 @ space2:.2e} (orthogonal)')

---
## 2. Linear Independence, Span, Basis, Dimension

**Definitions:**
- Vectors $v_1, \ldots, v_k$ are **linearly independent** if $\sum_i c_i v_i = 0 \implies c_i = 0\ \forall i$.
- Their **span** is $\{\sum_i c_i v_i : c_i \in \mathbb{R}\}$.
- A **basis** is a linearly independent spanning set for a subspace.
- All bases of a subspace have the same cardinality — that is the **dimension**.

**Computational test for independence:** The vectors $\{v_i\}$ are linearly dependent iff the matrix $V = [v_1 | \cdots | v_k]$ has $\det(V^T V) = 0$, equivalently the Gram matrix is singular. In NLA we detect near-dependence through small singular values.

In [ ]:
# Three cases: independent, dependent, near-dependent
cases = {
    'Independent': np.array([[1, 0], [0, 1], [1, 1]], dtype=float).T,
    'Dependent': np.array([[1, 0], [0, 1], [1, 1]], dtype=float).T * np.array([[1, 1, 2]]),  # v3 = v1+v2 exact
    'Near-dependent': np.array([[1, 0], [0, 1], [1+1e-8, 1+1e-8]], dtype=float).T
}
cases['Dependent'][:, 2] = cases['Dependent'][:, 0] + cases['Dependent'][:, 1]

print(f'{"Case":<20} {"Rank":<8} {"det(V^TV)":<18} {"Singular values"}')
print('-' * 65)
for name, V in cases.items():
    rank = np.linalg.matrix_rank(V)
    gram_det = np.linalg.det(V.T @ V)
    svs = np.linalg.svd(V, compute_uv=False)
    print(f'{name:<20} {rank:<8} {gram_det:<18.4e} {np.round(svs, 4)}')

In [ ]:
# Visualize span in 2D: what happens when we add a third vector to two others
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

v1 = np.array([1.5, 0.5])
v2 = np.array([0.3, 1.4])
v3_dep = v1 + v2           # linearly dependent
v3_indep = np.array([-0.5, 1.2])  # independent

def draw_span(ax, v1, v2, title, v3=None, label3=None):
    # Draw span as background fill
    ax.set_facecolor('#f8f8f8')
    ax.text(0, 0, 'span = $\\mathbb{R}^2$' if v3 is None or not np.isclose(np.linalg.matrix_rank(np.column_stack([v1, v2, v3])), 2) else '',
            ha='center', fontsize=9, color='gray')
    for v, color, label in [(v1, '#2E86AB', '$v_1$'), (v2, '#44BBA4', '$v_2$')]:
        ax.annotate('', xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=color, lw=2))
        ax.text(v[0]+0.05, v[1]+0.05, label, color=color, fontsize=13)
    if v3 is not None:
        color = '#E84855' if np.isclose(np.linalg.matrix_rank(np.column_stack([v1, v2, v3])), 2) else '#9B59B6'
        ax.annotate('', xy=v3, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=color, lw=2, ls='--'))
        ax.text(v3[0]+0.05, v3[1]+0.05, label3, color=color, fontsize=13)
    ax.axhline(0, color='k', lw=0.4)
    ax.axvline(0, color='k', lw=0.4)
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(-0.5, 2.5)
    ax.set_aspect('equal')
    ax.set_title(title)

draw_span(axes[0], v1, v2, r'$v_3 = v_1 + v_2$: dependent, rank stays 2', v3_dep, r'$v_3 = v_1+v_2$ (red, redundant)')
draw_span(axes[1], v1, v2, r'$v_3$ independent: still rank 2 (we are in $\mathbb{R}^2$)', v3_indep, r'$v_3$ (purple, also redundant in $\mathbb{R}^2$)')

plt.suptitle('Linear dependence: adding a vector does not increase span if dependent\n(in $\\mathbb{R}^2$, any 3 vectors are dependent)', fontweight='bold')
plt.tight_layout()
plt.savefig('span_visualization.png', bbox_inches='tight')
plt.show()

---
## 3. Linear Maps and Matrix Representations

A map $T: \mathbb{R}^n \to \mathbb{R}^m$ is **linear** iff $T(\alpha u + \beta v) = \alpha T(u) + \beta T(v)$.

**Key fact:** Once we fix a basis $\{e_j\}$ for $\mathbb{R}^n$ and $\{f_i\}$ for $\mathbb{R}^m$, every linear map is represented by a unique matrix $A$ where $A_{ij} = [T(e_j)]_i$ (the $i$-th component of $T$ applied to the $j$-th basis vector).

**Consequence for NLA:** A change of basis from $\{e_j\}$ to $\{q_j\}$ transforms the matrix representation as:
$$A \longrightarrow B = Q^{-1} A Q$$
where $Q$ is the change-of-basis matrix. Eigendecomposition, SVD, and all factorizations are fundamentally about choosing bases that make $A$'s representation simple (diagonal, triangular, etc.).

In [ ]:
# Visualize a linear map as geometric transformation of the unit square/circle
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])  # unit circle

transformations = {
    'Rotation $45°$': np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
                                 [np.sin(np.pi/4),  np.cos(np.pi/4)]]),
    'Shear': np.array([[1, 1.5], [0, 1]]),
    'Projection onto $x$-axis': np.array([[1, 0], [0, 0]]),
    'Stretch $\\sigma_1=3, \\sigma_2=0.5$': np.array([[3, 0], [0, 0.5]]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#2E86AB', '#44BBA4', '#E84855', '#F7A278']

for ax, (title, A), color in zip(axes, transformations.items(), colors):
    # Original circle
    ax.plot(circle[0], circle[1], 'k--', lw=1, alpha=0.4, label='original')
    # Transformed circle (becomes an ellipse)
    transformed = A @ circle
    ax.plot(transformed[0], transformed[1], color=color, lw=2, label='image')
    # Draw basis vectors and their images
    for e, ls in [(np.array([1, 0]), '-'), (np.array([0, 1]), '--')]:
        ax.annotate('', xy=e, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls=ls))
        Ae = A @ e
        ax.annotate('', xy=Ae, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2, ls=ls))
    ax.axhline(0, color='k', lw=0.4)
    ax.axvline(0, color='k', lw=0.4)
    lim = max(3.5, np.max(np.abs(transformed))*1.2)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.text(0.02, 0.98, f'det={np.linalg.det(A):.2f}', transform=ax.transAxes,
            va='top', fontsize=9, color='gray')

plt.suptitle('Linear maps acting on the unit circle — columns of $A$ are images of basis vectors',
             fontweight='bold')
plt.tight_layout()
plt.savefig('linear_maps.png', bbox_inches='tight')
plt.show()

In [ ]:
# Change of basis demonstration
# Represent the same linear map in two different bases
A = np.array([[3, 1], [0, 2]], dtype=float)  # standard basis representation

# New basis: columns of Q are the new basis vectors
Q = np.array([[1, 1], [1, -1]], dtype=float) / np.sqrt(2)  # orthonormal basis

# Representation in new basis: B = Q^{-1} A Q = Q^T A Q (since Q is orthogonal)
B = Q.T @ A @ Q

print('Same linear map, two basis representations:')
print(f'Standard basis representation A:\n{A}')
print(f'\nNew orthonormal basis Q:\n{np.round(Q, 4)}')
print(f'\nRepresentation in new basis B = Q^T A Q:\n{np.round(B, 6)}')
print(f'\nEigenvalues of A: {np.sort(np.linalg.eigvals(A)).real}')
print(f'Eigenvalues of B: {np.sort(np.linalg.eigvals(B)).real}  (same — similarity transformation)')
print(f'Trace A = {np.trace(A):.4f}, Trace B = {np.trace(B):.4f}  (invariant)')
print(f'Det A = {np.linalg.det(A):.4f}, Det B = {np.linalg.det(B):.4f}   (invariant)')

---
## 4. Inner Products and Norms

The **standard inner product** on $\mathbb{R}^n$: $\langle x, y \rangle = x^T y = \sum_{i=1}^n x_i y_i$.

Two vectors are **orthogonal** if $\langle x, y \rangle = 0$.

### Vector Norms

| Name | Formula | Key property |
|---|---|---|
| 1-norm | $\|x\|_1 = \sum |x_i|$ | Sparse solutions |
| 2-norm (Euclidean) | $\|x\|_2 = \sqrt{x^T x}$ | Orthogonality, geometry |
| $\infty$-norm | $\|x\|_\infty = \max |x_i|$ | Uniform convergence |
| $p$-norm | $\|x\|_p = (\sum |x_i|^p)^{1/p}$ | General family |

### Matrix Norms

The **induced (operator) $p$-norm**: $\|A\|_p = \sup_{x \neq 0} \frac{\|Ax\|_p}{\|x\|_p}$.

The critical case for NLA: $\|A\|_2 = \sigma_1(A)$ — the largest singular value.

The **Frobenius norm**: $\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\sum_i \sigma_i^2}$.

**Norm equivalence** in $\mathbb{R}^n$: for any two norms $\|\cdot\|_\alpha, \|\cdot\|_\beta$ there exist $c_1, c_2 > 0$ such that $c_1 \|x\|_\alpha \leq \|x\|_\beta \leq c_2 \|x\|_\alpha$.

In [ ]:
# Visualize unit balls in different norms in R^2
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
ps = [1, 2, 4, np.inf]
colors = ['#2E86AB', '#44BBA4', '#E84855', '#F7A278']
labels = ['$p=1$ (diamond)', '$p=2$ (circle)', '$p=4$', '$p=\\infty$ (square)']

theta = np.linspace(0, 2*np.pi, 2000)

for ax, p, color, label in zip(axes, ps, colors, labels):
    if p == np.inf:
        # unit square
        x_vals = np.array([-1, 1, 1, -1, -1])
        y_vals = np.array([-1, -1, 1, 1, -1])
    else:
        # parameterize: find y given x on the unit p-ball
        x_half = np.linspace(-1, 1, 1000)
        y_half = (1 - np.abs(x_half)**p)**(1/p)
        x_vals = np.concatenate([x_half, x_half[::-1]])
        y_vals = np.concatenate([y_half, -y_half[::-1]])

    ax.fill(x_vals, y_vals, alpha=0.3, color=color)
    ax.plot(x_vals, y_vals, color=color, lw=2)
    ax.axhline(0, color='k', lw=0.4)
    ax.axvline(0, color='k', lw=0.4)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(label)

plt.suptitle(r'Unit balls $\{x \in \mathbb{R}^2 : \|x\|_p = 1\}$ for various $p$', fontweight='bold')
plt.tight_layout()
plt.savefig('norm_balls.png', bbox_inches='tight')
plt.show()

In [ ]:
# Induced 2-norm = largest singular value: geometric verification
A = np.array([[3, 1], [1, 2]], dtype=float)
U, S, Vt = np.linalg.svd(A)

# The induced 2-norm = max ||Ax||_2 over unit 2-sphere
# Verify by sampling
N = 100000
x_rand = rng.standard_normal((2, N))
x_rand /= np.linalg.norm(x_rand, axis=0)  # normalize to unit sphere
norms_Ax = np.linalg.norm(A @ x_rand, axis=0)

print(f'A =\n{A}')
print(f'\nSingular values: σ₁={S[0]:.6f}, σ₂={S[1]:.6f}')
print(f'||A||₂ = σ₁ = {S[0]:.6f}')
print(f'Max ||Ax||₂ over 10^5 unit vectors = {norms_Ax.max():.6f}  (matches)')
print(f'||A||_F = sqrt(σ₁²+σ₂²) = {np.sqrt(S[0]**2+S[1]**2):.6f}')
print(f'np.linalg.norm(A, "fro") = {np.linalg.norm(A, "fro"):.6f}')

# The maximizing direction is the first right singular vector v_1
v1 = Vt[0]
print(f'\nMaximizing direction v₁ = {v1}')
print(f'||A v₁||₂ = {np.linalg.norm(A @ v1):.6f} = σ₁')

In [ ]:
# Visualize: A maps unit circle to an ellipse; semi-axes are σ₁,σ₂ aligned with U
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

theta_vals = np.linspace(0, 2*np.pi, 300)
unit_circ = np.vstack([np.cos(theta_vals), np.sin(theta_vals)])
ellipse = A @ unit_circ

# Left: input space with right singular vectors
axes[0].plot(unit_circ[0], unit_circ[1], 'k-', lw=1.5, label='unit circle (input)')
for i, (v, color) in enumerate(zip(Vt, ['#2E86AB', '#E84855'])):
    axes[0].annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    axes[0].text(v[0]+0.05, v[1]+0.05, f'$v_{i+1}$', color=color, fontsize=14)
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal'); axes[0].set_title('Input space: right singular vectors')
axes[0].axhline(0, color='k', lw=0.4); axes[0].axvline(0, color='k', lw=0.4)

# Right: output space with left singular vectors scaled by singular values
axes[1].plot(ellipse[0], ellipse[1], color='#44BBA4', lw=2, label='image ellipse')
for i, (u, s, color) in enumerate(zip(U.T, S, ['#2E86AB', '#E84855'])):
    axes[1].annotate('', xy=s*u, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    axes[1].text(s*u[0]+0.05, s*u[1]+0.05, f'$\\sigma_{i+1} u_{i+1}$', color=color, fontsize=12)
axes[1].set_xlim(-4.5, 4.5); axes[1].set_ylim(-4.5, 4.5)
axes[1].set_aspect('equal'); axes[1].set_title(f'Output space: image ellipse, semi-axes $\\sigma_1={S[0]:.2f}, \\sigma_2={S[1]:.2f}$')
axes[1].axhline(0, color='k', lw=0.4); axes[1].axvline(0, color='k', lw=0.4)

plt.suptitle(r'$A$ maps unit circle → ellipse. $\|A\|_2 = \sigma_1 =$ length of longest semi-axis.', fontweight='bold')
plt.tight_layout()
plt.savefig('svd_geometry.png', bbox_inches='tight')
plt.show()

---
## 5. Orthogonal Projections

Let $S$ be a subspace of $\mathbb{R}^m$ with orthonormal basis $Q = [q_1 | \cdots | q_r]$. The **orthogonal projector** onto $S$ is:
$$P = Q Q^T$$

**Properties:** $P^2 = P$ (idempotent), $P^T = P$ (symmetric), $I - P$ is the projector onto $S^\perp$.

**Geometric meaning:** For any $b \in \mathbb{R}^m$, $Pb$ is the unique vector in $S$ closest to $b$:
$$Pb = \arg\min_{v \in S} \|b - v\|_2$$

**Proof:** Write $b = Pb + (b - Pb)$. For any $v \in S$:
$$\|b - v\|^2 = \|(Pb - v) + (b - Pb)\|^2 = \|Pb - v\|^2 + \|b - Pb\|^2$$
since $Pb - v \in S$ and $b - Pb \in S^\perp$ are orthogonal. This is minimized when $v = Pb$. $\square$

This is the geometric engine behind **least squares**: $\hat{x} = (A^T A)^{-1} A^T b$ gives $A\hat{x} = Pb$ where $P$ projects onto $\mathcal{C}(A)$.

In [ ]:
# 1. Verify projector properties algebraically
# Basis for a subspace (a plane in R^3)
Q_raw = np.array([[1, 0], [1, 1], [0, 1]], dtype=float)
Q, _ = np.linalg.qr(Q_raw)  # orthonormalize
Q = Q[:, :2]  # keep first 2 columns

P = Q @ Q.T  # projector onto the plane

print('Projector P = QQ^T:')
print(np.round(P, 6))
print(f'\nP^2 == P? max|P^2 - P| = {np.max(np.abs(P @ P - P)):.2e}')
print(f'P^T == P? max|P^T - P| = {np.max(np.abs(P.T - P)):.2e}')
print(f'Eigenvalues of P: {np.round(np.sort(np.linalg.eigvals(P).real)[::-1], 4)}')
print('  (Should be: r ones, m-r zeros. r=2, m=3 → [1, 1, 0])')

# 2. Projection minimizes distance
b = np.array([3, 1, 2], dtype=float)
Pb = P @ b
print(f'\nb = {b}')
print(f'Pb = {np.round(Pb, 4)}  (projection onto plane)')
print(f'Residual r = b - Pb = {np.round(b - Pb, 4)}')
print(f'r ⊥ C(Q)? Q^T r = {np.round(Q.T @ (b - Pb), 10)}  (should be 0)')

In [ ]:
# Visualize projection in 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Draw the projection plane (span of Q columns)
xx, yy = np.meshgrid(np.linspace(-1, 4, 20), np.linspace(-1, 4, 20))
# The plane normal is the left null space vector of Q
_, _, Vt_plane = np.linalg.svd(Q.T)
n = Vt_plane[-1]  # normal to the plane
zz = (-n[0] * xx - n[1] * yy) / n[2]
ax.plot_surface(xx, yy, zz, alpha=0.15, color='#2E86AB')

# Draw b and its projection Pb
ax.quiver(0, 0, 0, *b, color='#E84855', lw=2, arrow_length_ratio=0.1, label=r'$b$')
ax.quiver(0, 0, 0, *Pb, color='#2E86AB', lw=2.5, arrow_length_ratio=0.1, label=r'$Pb$ (projection)')
# Draw residual
ax.quiver(*Pb, *(b-Pb), color='#44BBA4', lw=2, arrow_length_ratio=0.15, label=r'$b - Pb$ (residual, $\perp$ plane)')

# Draw right angle marker
eps = 0.12
orth_mark = Pb + eps * (b - Pb) / np.linalg.norm(b - Pb)
ax.scatter(*orth_mark, color='#44BBA4', s=20)

ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title('Orthogonal Projection onto a Plane in $\\mathbb{R}^3$\n$Pb$ is the closest point in the subspace to $b$', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 3.5); ax.set_ylim(0, 2); ax.set_zlim(0, 3)
plt.tight_layout()
plt.savefig('projection_3d.png', bbox_inches='tight')
plt.show()

In [ ]:
# Least squares as projection: overdetermined system Ax = b
# Fit a quadratic to noisy data
t = np.linspace(0, 1, 20)
b_noisy = 2 + 3*t - 5*t**2 + 0.3 * rng.standard_normal(20)

# Vandermonde matrix: columns are 1, t, t^2
A_ls = np.column_stack([np.ones_like(t), t, t**2])

# Least squares solution via normal equations: x = (A^T A)^{-1} A^T b
# Equivalently: x = A^+ b (pseudoinverse)
x_ls, residuals, rank, sv = np.linalg.lstsq(A_ls, b_noisy, rcond=None)

# The projection Pb = A x_ls
Pb_ls = A_ls @ x_ls
r_ls = b_noisy - Pb_ls

# Verify residual is orthogonal to C(A)
print(f'Least squares solution: x = {np.round(x_ls, 4)}')
print(f'True coefficients:          [2.0000, 3.0000, -5.0000]')
print(f'\nA^T r (should be ~0): {np.round(A_ls.T @ r_ls, 10)}')
print(f'||r||₂ = {np.linalg.norm(r_ls):.6f}  (minimum possible residual norm)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_fine = np.linspace(0, 1, 200)
A_fine = np.column_stack([np.ones_like(t_fine), t_fine, t_fine**2])

axes[0].scatter(t, b_noisy, color='#E84855', zorder=5, label='data $b$')
axes[0].plot(t_fine, A_fine @ x_ls, color='#2E86AB', lw=2, label=r'fit $A\hat{x} = Pb$')
axes[0].plot(t_fine, 2 + 3*t_fine - 5*t_fine**2, 'k--', lw=1.5, alpha=0.5, label='true curve')
for ti, bi, pbi in zip(t, b_noisy, Pb_ls):
    axes[0].plot([ti, ti], [bi, pbi], color='#44BBA4', lw=0.8, alpha=0.7)
axes[0].set_title('Least Squares Fit = Projection onto $\\mathcal{C}(A)$')
axes[0].legend()

# Show residual is orthogonal to all columns of A
axes[1].bar(range(3), np.abs(A_ls.T @ r_ls), color=['#2E86AB', '#44BBA4', '#E84855'])
axes[1].set_xticks(range(3)); axes[1].set_xticklabels(['$a_1^T r$', '$a_2^T r$', '$a_3^T r$'])
axes[1].set_ylabel('|value|')
axes[1].set_title('Residual $r$ is orthogonal to all columns of $A$\n(Normal equations satisfied)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('least_squares_projection.png', bbox_inches='tight')
plt.show()

---
## 6. Eigenvalues and the Spectral Theorem

A scalar $\lambda$ and nonzero vector $v$ satisfy $Av = \lambda v$ — $v$ is an **eigenvector**, $\lambda$ the **eigenvalue**.

**Spectral Theorem:** If $A \in \mathbb{R}^{n \times n}$ is symmetric ($A = A^T$), then:
1. All eigenvalues are real.
2. Eigenvectors for distinct eigenvalues are orthogonal.
3. $A$ has an orthonormal eigenbasis: $A = Q \Lambda Q^T$ where $Q$ is orthogonal, $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

**Why symmetric matrices dominate NLA:** $A^T A$, $A A^T$, covariance matrices, stiffness matrices, Hessians — all symmetric. The spectral theorem guarantees diagonalizability with an orthonormal basis. No Jordan blocks, no complex arithmetic.

**Rayleigh quotient:** $R(A, x) = \frac{x^T A x}{x^T x}$, which satisfies $\lambda_{\min} \leq R(A,x) \leq \lambda_{\max}$, with equality at the corresponding eigenvectors. This is the variational principle underlying power iteration and Krylov methods.

In [ ]:
# Construct a symmetric matrix and verify spectral decomposition
rng2 = np.random.default_rng(7)
M = rng2.standard_normal((4, 4))
A_sym = M @ M.T  # symmetric positive definite

eigvals, eigvecs = np.linalg.eigh(A_sym)  # eigh: for symmetric, returns real, sorted
eigvals = eigvals[::-1]  # sort descending
eigvecs = eigvecs[:, ::-1]

print('Symmetric matrix A = M M^T:')
print(np.round(A_sym, 4))
print(f'\nEigenvalues: {np.round(eigvals, 6)}')
print(f'All real: {np.all(np.isreal(eigvals))}')

# Verify orthonormality of eigenvectors
print(f'\nQ^T Q (should be I):')
print(np.round(eigvecs.T @ eigvecs, 10))

# Verify spectral decomposition A = Q Λ Q^T
A_reconstructed = eigvecs @ np.diag(eigvals) @ eigvecs.T
print(f'\n||A - Q Λ Q^T||_F = {np.linalg.norm(A_sym - A_reconstructed):.2e}')

# Rayleigh quotient on random unit vectors
x_test = rng2.standard_normal((4, 10000))
x_test /= np.linalg.norm(x_test, axis=0)
RQ = np.einsum('ij,ij->j', x_test, A_sym @ x_test)  # x^T A x for each column
print(f'\nRayleigh quotient bounds:')
print(f'  λ_min = {eigvals[-1]:.6f},  min R(A,x) over samples = {RQ.min():.6f}')
print(f'  λ_max = {eigvals[0]:.6f},  max R(A,x) over samples = {RQ.max():.6f}')

In [ ]:
# Visualize: eigenvectors as principal axes of the quadratic form x^T A x = const (ellipse in 2D)
A_2d = np.array([[3, 1.5], [1.5, 1.5]])  # symmetric 2x2
eigvals_2d, eigvecs_2d = np.linalg.eigh(A_2d)
eigvals_2d = eigvals_2d[::-1]; eigvecs_2d = eigvecs_2d[:, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: quadratic form level sets
x_grid = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(x_grid, x_grid)
Z = A_2d[0,0]*X**2 + 2*A_2d[0,1]*X*Y + A_2d[1,1]*Y**2
c = axes[0].contourf(X, Y, Z, levels=20, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(c, ax=axes[0])
axes[0].contour(X, Y, Z, levels=[0.5, 1, 2, 4], colors='white', alpha=0.6, linewidths=1)

for i, (ev, lam, color) in enumerate(zip(eigvecs_2d.T, eigvals_2d, ['#2E86AB', '#E84855'])):
    scale = 1.0 / np.sqrt(lam)
    axes[0].annotate('', xy=scale*ev, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].annotate('', xy=-scale*ev, xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color=color, lw=3))
    axes[0].text(scale*ev[0]+0.05, scale*ev[1]+0.05,
                 f'$q_{i+1}$, $\\lambda_{i+1}={lam:.2f}$', color=color, fontsize=11)
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].set_aspect('equal')
axes[0].set_title('Level sets of $x^T A x$\nEigenvectors = principal axes of ellipses')
axes[0].axhline(0, color='w', lw=0.5); axes[0].axvline(0, color='w', lw=0.5)

# Right: Rayleigh quotient as function of angle
theta_rq = np.linspace(0, 2*np.pi, 500)
x_unit = np.vstack([np.cos(theta_rq), np.sin(theta_rq)])
rq_vals = np.einsum('ij,ij->j', x_unit, A_2d @ x_unit)
axes[1].plot(np.degrees(theta_rq), rq_vals, color='#2E86AB', lw=2)
axes[1].axhline(eigvals_2d[0], color='#E84855', ls='--', lw=1.5, label=f'$\\lambda_1 = {eigvals_2d[0]:.3f}$')
axes[1].axhline(eigvals_2d[1], color='#44BBA4', ls='--', lw=1.5, label=f'$\\lambda_2 = {eigvals_2d[1]:.3f}$')
axes[1].set_xlabel('angle (degrees)')
axes[1].set_ylabel('$R(A, x(\\theta)) = x^T A x$')
axes[1].set_title('Rayleigh quotient over unit circle\nBounded by $[\\lambda_{\\min}, \\lambda_{\\max}]$')
axes[1].legend()
# Mark eigenvector angles
for i, ev in enumerate(eigvecs_2d.T):
    ang = np.degrees(np.arctan2(ev[1], ev[0])) % 360
    axes[1].axvline(ang, color='gray', ls=':', lw=1)

plt.suptitle('Spectral Theorem: symmetric $A$ has orthogonal eigenbasis = principal axes of quadratic form',
             fontweight='bold')
plt.tight_layout()
plt.savefig('spectral_theorem.png', bbox_inches='tight')
plt.show()

---
## 7. Rank and Near-Rank-Deficiency

The **rank** of $A$ is $r = \dim \mathcal{C}(A) = \dim \mathcal{C}(A^T)$, equivalently the number of nonzero singular values.

**Rank deficiency** ($r < \min(m,n)$) has two distinct effects:
- Theoretical: $Ax = b$ has either no solution or infinitely many.
- Numerical: *near* rank deficiency — small but nonzero singular values — leads to catastrophic amplification of errors. This is why we care about **condition number** $\kappa(A) = \sigma_1 / \sigma_r$.

**The rank-revealing SVD:** $A = U \Sigma V^T$ where $\Sigma = \text{diag}(\sigma_1 \geq \cdots \geq \sigma_r > 0 = \sigma_{r+1} = \cdots)$. The SVD simultaneously reveals:
- The rank (number of nonzero $\sigma_i$)
- An orthonormal basis for each of the four fundamental subspaces
- The condition number $\kappa = \sigma_1 / \sigma_r$
- The best rank-$k$ approximation (Eckart-Young theorem)

In [ ]:
# Condition number and its effect on linear system solving
# Hilbert matrix: notoriously ill-conditioned
def hilbert(n):
    return np.array([[1/(i+j+1) for j in range(n)] for i in range(n)])

print(f'{"n":<6} {"κ(H_n)":<20} {"log₁₀(κ)":<15} {"Relative error ||Ax-b||/||b||"}')
print('-' * 70)
for n in [3, 5, 8, 10, 12]:
    H = hilbert(n)
    x_true = np.ones(n)
    b = H @ x_true
    x_computed = np.linalg.solve(H, b)
    rel_err = np.linalg.norm(x_computed - x_true) / np.linalg.norm(x_true)
    kappa = np.linalg.cond(H)
    print(f'{n:<6} {kappa:<20.4e} {np.log10(kappa):<15.2f} {rel_err:.4e}')

In [ ]:
# Eckart-Young theorem: SVD gives best low-rank approximation
# ||A - A_k||_2 = σ_{k+1}  and  ||A - A_k||_F = sqrt(σ_{k+1}^2 + ... + σ_r^2)

# Load a "matrix" to compress: grayscale image-like random matrix
rng3 = np.random.default_rng(13)
# Construct a low-rank + noise matrix
m, n = 80, 80
r_true = 5
L = rng3.standard_normal((m, r_true)) @ rng3.standard_normal((r_true, n))  # rank-5 signal
noise = 0.5 * rng3.standard_normal((m, n))
A_noisy = L + noise

U_n, S_n, Vt_n = np.linalg.svd(A_noisy)

# Compute approximation errors
ks = range(1, 20)
errors_2 = [S_n[k] for k in ks]          # ||A - A_k||_2 = σ_{k+1}
errors_F = [np.sqrt(np.sum(S_n[k:]**2)) for k in ks]  # ||A - A_k||_F

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot singular value spectrum
axes[0].semilogy(range(1, len(S_n)+1), S_n, 'o-', color='#2E86AB', ms=4)
axes[0].axvline(r_true, color='#E84855', ls='--', lw=2, label=f'true rank = {r_true}')
axes[0].set_xlabel('index $i$'); axes[0].set_ylabel('$\\sigma_i$')
axes[0].set_title('Singular Value Spectrum\n(gap reveals true rank)')
axes[0].legend()

# Plot approximation errors
axes[1].semilogy(list(ks), errors_2, 's-', color='#44BBA4', label=r'$\|A - A_k\|_2 = \sigma_{k+1}$')
axes[1].semilogy(list(ks), errors_F, '^-', color='#E84855', label=r'$\|A - A_k\|_F$')
axes[1].axvline(r_true, color='gray', ls=':', lw=2, label=f'true rank = {r_true}')
axes[1].set_xlabel('rank $k$'); axes[1].set_ylabel('approximation error')
axes[1].set_title('Eckart-Young: Best Rank-$k$ Approx Error')
axes[1].legend(fontsize=9)

# Show visual reconstruction at different ranks
rank_show = 5
A_k = U_n[:, :rank_show] @ np.diag(S_n[:rank_show]) @ Vt_n[:rank_show, :]
vmin, vmax = A_noisy.min(), A_noisy.max()
axes[2].imshow(A_k, cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
axes[2].set_title(f'Rank-{rank_show} approximation\n(captures signal, removes noise)')
axes[2].set_xlabel('col index'); axes[2].set_ylabel('row index')

plt.suptitle('Rank and the SVD: Eckart-Young theorem and rank detection via singular value gap',
             fontweight='bold')
plt.tight_layout()
plt.savefig('rank_svd.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary: Tie all concepts together via SVD
print('='*60)
print('SVD UNIFIES ALL SEVEN PRELIMINARIES')
print('='*60)
A_summary = np.array([[1, 2, 0],
                       [0, 1, 1],
                       [1, 3, 1],
                       [2, 1, -1]], dtype=float)

U_s, S_s, Vt_s = np.linalg.svd(A_summary, full_matrices=True)
r_s = np.sum(S_s > 1e-10)

print(f'\nA: {A_summary.shape[0]}x{A_summary.shape[1]}, rank = {r_s}')
print(f'\n1. FOUR SUBSPACES:')
print(f'   C(A):  dim={r_s}, basis = U[:,:{r_s}]')
print(f'   N(A^T): dim={A_summary.shape[0]-r_s}, basis = U[:,{r_s}:]')
print(f'   C(A^T): dim={r_s}, basis = Vt[:{r_s},:].T')
print(f'   N(A):  dim={A_summary.shape[1]-r_s}, basis = Vt[{r_s}:,:].T')
print(f'\n2. BASIS/DIMENSION: column space dim = row space dim = {r_s}')
print(f'\n3. LINEAR MAP: A = U Σ V^T decomposes into rotate × stretch × rotate')
print(f'\n4. NORMS: ||A||₂ = σ₁ = {S_s[0]:.4f},  ||A||_F = {np.linalg.norm(A_summary, "fro"):.4f} = sqrt(Σσᵢ²)')
print(f'\n5. PROJECTION: P_col = U[:,:{r_s}] @ U[:,:{r_s}].T (projector onto C(A))')
P_col = U_s[:, :r_s] @ U_s[:, :r_s].T
b_test = np.array([1, 0, 0, 1], dtype=float)
print(f'   ||Pb - b||₂ = {np.linalg.norm(P_col @ b_test - b_test):.4f} (residual from projection)')
print(f'\n6. EIGENVALUES: A^T A has eigenvalues = σᵢ² = {np.round(S_s**2, 4)}')
print(f'   eigvecs of A^T A = rows of V^T (right singular vectors)')
print(f'\n7. RANK/CONDITION: κ(A) = σ₁/σᵣ = {S_s[0]/S_s[r_s-1]:.4f}  (well-conditioned if ~1)')

---
## Summary

The seven preliminaries form a coherent structure, not a list:

```
Vector spaces + linear maps
       ↓
Span, independence, basis, dimension
       ↓
Inner products → orthogonality → norms
       ↓
Orthogonal projections  ←→  Least squares
       ↓
Eigenvalues/Spectral Theorem  ←→  Symmetric systems
       ↓
Rank, singular values, condition number
       ↓
      SVD  ←— unifies all of the above
```

From here, Trefethen & Bau Chapter 1-3 (QR/Gram-Schmidt), Chapter 4-5 (least squares), and Chapter 6+ (eigenvalue algorithms) all build directly on this foundation.